In [3]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / (1024 ** 3), 2
        ),
        "GB"
    )

CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [4]:
%pip install -q -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 91.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.7 MB/s eta 0:00:00


In [5]:
from huggingface_hub import login

login()

In [6]:
from google.colab import userdata
from huggingface_hub import HfApi

token = userdata.get("HF_TOKEN")

api = HfApi(token=token)

user = api.whoami()
print("Authenticated successfully")
print("Hugging Face username:", user["name"])

Authenticated successfully
Hugging Face username: [redacted]


In [7]:
from huggingface_hub import hf_hub_download

model_id = "NCAIR1/N-ATLaS"

config_path = hf_hub_download(
    repo_id=model_id,
    filename="config.json",
    token=token
)

print("N-ATLaS repository access successful!")
print("Configuration downloaded.")

config.json:   0%|          | 0.00/875 [00:00<?, ?B/s]

N-ATLaS repository access successful!
Configuration downloaded.


In [8]:

# ============================================================
# NaijaCyber AI - STEP D
# Load N-ATLaS using 4-bit quantisation
# ============================================================

import torch

from google.colab import userdata
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

# 1. Retrieve Hugging Face token securely
HF_TOKEN = userdata.get("HF_TOKEN")

# 2. Official model repository
MODEL_ID = "NCAIR1/N-ATLaS"

# 3. Verify GPU availability
if not torch.cuda.is_available():
    raise RuntimeError(
        "No NVIDIA GPU detected. In Colab, select "
        "Runtime > Change runtime type > T4 GPU."
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_memory = (
    torch.cuda.get_device_properties(0).total_memory
    / (1024 ** 3)
)

print(f"GPU detected: {gpu_name}")
print(f"Total GPU memory: {gpu_memory:.2f} GB")

# 4. Configure memory-efficient 4-bit loading
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

# 5. Load tokenizer
print("\nLoading N-ATLaS tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

# 6. Load model
print("Loading N-ATLaS model. This may take several minutes...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
)

# 7. Set model to evaluation mode
model.eval()

# 8. Configure padding
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

# 9. Verify model loading
print("\n====================================")
print("N-ATLaS LOADED SUCCESSFULLY")
print("====================================")
print(f"Model: {MODEL_ID}")
print(f"GPU: {gpu_name}")
print("Quantisation: 4-bit NF4")
print(f"Model class: {model.__class__.__name__}")

allocated = torch.cuda.memory_allocated() / (1024 ** 3)
print(f"GPU memory allocated: {allocated:.2f} GB")


GPU detected: Tesla T4
Total GPU memory: 14.56 GB

Loading N-ATLaS tokenizer...


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


N-ATLaS LOADED SUCCESSFULLY
Model: NCAIR1/N-ATLaS
GPU: Tesla T4
Quantisation: 4-bit NF4
Model class: LlamaForCausalLM
GPU memory allocated: 5.32 GB


In [9]:

# ============================================================
# NaijaCyber AI - STEP E
# First real N-ATLaS cybersecurity inference
# ============================================================

import time
import torch
from datetime import datetime

# 1. Prepare the cybersecurity tutor prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are NaijaCyber AI, a cybersecurity education "
            "assistant designed for Nigerian beginners. "
            "Explain cybersecurity concepts clearly and accurately. "
            "Use simple language and realistic Nigerian examples. "
            "Never request passwords, PINs, or OTPs."
        )
    },
    {
        "role": "user",
        "content": (
            "Explain phishing in simple English using an example "
            "involving a Nigerian banking customer. "
            "Keep your explanation under 120 words."
        )
    }
]

# 2. Format messages using the N-ATLaS chat template
# The model documentation specifies a date_string argument.
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    date_string=datetime.now().strftime("%d %b %Y")
)

# 3. Tokenise the prompt
inputs = tokenizer(
    formatted_prompt,
    return_tensors="pt",
    add_special_tokens=False
).to(model.device)

# 4. Generate the response
print("Generating response using N-ATLaS...\n")

start_time = time.perf_counter()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=180,
        do_sample=False,
        repetition_penalty=1.12,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True
    )

elapsed = time.perf_counter() - start_time

# 5. Extract only the newly generated tokens
generated_tokens = outputs[0][
    inputs["input_ids"].shape[-1]:
]

response = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
).strip()

# 6. Display results
print("=" * 60)
print("NAIJACYBER AI - N-ATLaS INFERENCE RESULT")
print("=" * 60)

print("\nModel:", MODEL_ID)
print("Language: English")
print("Topic: Phishing")
print("\nMODEL RESPONSE:\n")
print(response)

print("\n" + "=" * 60)
print("INFERENCE METRICS")
print("=" * 60)

token_count = generated_tokens.shape[-1]

print(f"Generated tokens: {token_count}")
print(f"Inference time: {elapsed:.2f} seconds")

if elapsed > 0:
    print(f"Generation speed: {token_count / elapsed:.2f} tokens/sec")

print("\nInference completed.")


Generating response using N-ATLaS...



[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


NAIJACYBER AI - N-ATLaS INFERENCE RESULT

Model: NCAIR1/N-ATLaS
Language: English
Topic: Phishing

MODEL RESPONSE:

Phishing is when someone tries to trick you into giving them personal information like bank details. Imagine a person sending you an email that looks like it's from your bank, asking you to click on a link to "verify" your account. If you click, you might go to a fake website that steals your login info. Don't fall for it! Always check the sender's email address and avoid clicking suspicious links. Your bank won’t ask you sensitive questions via email.

INFERENCE METRICS
Generated tokens: 96
Inference time: 8.56 seconds
Generation speed: 11.21 tokens/sec

Inference completed.


In [10]:

# =====================================================
# NaijaCyber AI - STEP F
# Multilingual N-ATLaS Cybersecurity Evaluation
# =====================================================

import time
import json
import torch
from datetime import datetime

evaluation_prompts = {
    "English": (
        "Explain phishing to a cybersecurity beginner "
        "in simple English. Use a Nigerian banking example. "
        "Keep your response under 120 words."
    ),
    "Hausa": (
        "Explain phishing to a cybersecurity beginner "
        "entirely in Hausa. Use a Nigerian banking example. "
        "Keep your response under 120 words."
    ),
    "Igbo": (
        "Explain phishing to a cybersecurity beginner "
        "entirely in Igbo. Use a Nigerian banking example. "
        "Keep your response under 120 words."
    ),
    "Yoruba": (
        "Explain phishing to a cybersecurity beginner "
        "entirely in Yoruba. Use a Nigerian banking example. "
        "Keep your response under 120 words."
    )
}

results = []

for language, question in evaluation_prompts.items():

    messages = [
        {
            "role": "system",
            "content": (
                "You are NaijaCyber AI, a cybersecurity "
                "education assistant for Nigerian beginners. "
                "Respond only in the requested language. "
                "Provide accurate and safe explanations."
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        date_string=datetime.now().strftime("%d %b %Y")
    )

    inputs = tokenizer(
        formatted_prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=180,
            do_sample=False,
            repetition_penalty=1.12,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start_time

    generated_tokens = output[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    token_count = generated_tokens.shape[-1]

    result = {
        "language": language,
        "model": MODEL_ID,
        "prompt": question,
        "response": response,
        "tokens": int(token_count),
        "latency_seconds": round(elapsed, 2),
        "tokens_per_second": round(
            token_count / elapsed, 2
        ) if elapsed > 0 else 0
    }

    results.append(result)

    print("\n" + "=" * 60)
    print(f"LANGUAGE: {language}")
    print("=" * 60)
    print(response)
    print(f"\nLatency: {elapsed:.2f}s")
    print(f"Tokens/sec: {result['tokens_per_second']}")

# Save the raw evaluation evidence
with open(
    "naijacyber_multilingual_results.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        results,
        file,
        indent=2,
        ensure_ascii=False
    )

print("\nEvaluation completed.")
print("Saved: naijacyber_multilingual_results.json")



LANGUAGE: English
Phishing is like when someone sends you an email that looks like it's from your bank, asking you to give them your secret details. But, it's actually a trick by bad people trying to steal your money! Real banks won't ask you for personal stuff via email. So, always be careful and check if the message is real before sharing any information.

Latency: 4.77s
Tokens/sec: 15.51

LANGUAGE: Hausa
Phishing wani nau'in satar bayanan sirri ne ta hanyar yaudara da kuma yin amfani da zamba ta imel ko wasu kafofin sadarwa na zamani. Misali, bankunan Najeriya suna aika saƙonnin imel don tabbatar da bayananku, amma masu sata na iya yin kama da waɗannan. Idan ka danna mahaɗin da ba daidai ba, za su iya samun bayanan asusunka. Don guje wa wannan, duba adireshin imel don duk wani rashin daidaito kamar manyan haruffa ko rubutattun kalmomi kafin shiga. Ka kiyaye bayananka cik

Latency: 11.83s
Tokens/sec: 15.21

LANGUAGE: Igbo
Ịfịkọta bụ ụdị aghụghọ cyber nke ndị ohi na-eji izu ohi ozi n

In [11]:

# NaijaCyber AI - STEP G
# Retest N-ATLaS with truncation monitoring

import time
import torch
from datetime import datetime

TEST_LANGUAGES = {
    "Hausa": "Hausa",
    "Igbo": "Igbo",
    "Yoruba": "Yoruba"
}

def test_language(language, max_tokens=320):
    messages = [
        {
            "role": "system",
            "content": (
                "You are NaijaCyber AI, a cybersecurity "
                "tutor for Nigerian beginners. Explain "
                "security concepts accurately and clearly. "
                "Use only the requested language."
            )
        },
        {
            "role": "user",
            "content": (
                f"In {language}, explain phishing in 3 "
                "short sentences. Explain what it is, "
                "give a Nigerian banking example, and "
                "explain how to stay safe. "
                "Finish your answer completely."
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        date_string=datetime.now().strftime("%d %b %Y")
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        add_special_tokens=False
    ).to(model.device)

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            repetition_penalty=1.12,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True
        )

    torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    new_tokens = output[0][
        inputs["input_ids"].shape[-1]:
    ]

    response = tokenizer.decode(
        new_tokens,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    ).strip()

    eos_ids = model.generation_config.eos_token_id
    if eos_ids is None:
        eos_ids = [tokenizer.eos_token_id]
    elif isinstance(eos_ids, int):
        eos_ids = [eos_ids]

    ended_with_eos = (
        int(new_tokens[-1]) in eos_ids
        if new_tokens.numel() > 0
        else False
    )

    hit_limit = (
        len(new_tokens) >= max_tokens
        and not ended_with_eos
    )

    print(f"\n{'=' * 55}")
    print(f"LANGUAGE: {language}")
    print(f"{'=' * 55}")
    print(response)
    print(f"\nGenerated tokens: {len(new_tokens)}")
    print(f"Latency: {elapsed:.2f} seconds")
    print(f"Ended with EOS: {ended_with_eos}")
    print(f"Hit generation limit: {hit_limit}")

    return {
        "language": language,
        "response": response,
        "tokens": len(new_tokens),
        "latency": elapsed,
        "ended_with_eos": ended_with_eos,
        "hit_limit": hit_limit
    }


retest_results = []

for language in TEST_LANGUAGES:
    result = test_language(language)
    retest_results.append(result)



LANGUAGE: Hausa
Phishing wani nau'in zamba ne da ke amfani da saƙonnin imel ko wasu hanyoyin sadarwa na zamantakewa don yaudara masu amfani su bayyana bayanan sirri kamar bayanan banki ko lambobin ID. Misali a Najeriya, bankuna suna yawan zama abubuwan da aka yi niyya ga hare-haren satar bayanai ta hanyar phishing, inda masu cin zarafi za su iya aiko muku da saƙon da ya nuna cewa daga bankinku ne kuma yana buƙatar inganta bayanan ku. Don kasancewa lafiya, koyaushe duba adiresoshin imel don tabbatar da an samo shi daga tushen gaskiya kuma kar a bayar da bayanan sirrinku ga kowa sai dai idan kun tabbata cikakke game da asalin mai neman.

Generated tokens: 224
Latency: 14.68 seconds
Ended with EOS: True
Hit generation limit: False

LANGUAGE: Igbo
Phishing bụ aghụghọ nke ndị mmadụ na-anwa iji ozi nkeonwe ha dị ka akara kredit ma ọ bụ paswọọdụ banki wdg site n'ime ime email ụgha, ozi ederede, ma ọ bụ weebụsaịtị adịgboroja. Dịka ọmụmaatụ, onye scammer nwere ike izipu gị ozi-e nke yiri ka ọ 

In [12]:
%pip install -q fastapi uvicorn pyngrok

In [13]:

# =========================================================
# NAIJACYBER AI - COLAB INFERENCE SERVER
# Real N-ATLaS model with authenticated API access
# =========================================================

import os
import time
import hmac
import threading
import torch
import uvicorn

from datetime import datetime
from google.colab import userdata
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel, Field

# ---------------------------------------------------------
# 1. Validate loaded N-ATLaS model
# ---------------------------------------------------------

MODEL_ID = "NCAIR1/N-ATLaS"

if "model" not in globals() or "tokenizer" not in globals():
    raise RuntimeError(
        "N-ATLaS is not loaded. Run Step D again."
    )

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is available in this Colab runtime."
    )

# ---------------------------------------------------------
# 2. Retrieve API authentication secret
# ---------------------------------------------------------

API_KEY = userdata.get("NATLAS_API_KEY")

if not API_KEY or len(API_KEY) < 32:
    raise RuntimeError(
        "NATLAS_API_KEY must be at least "
        "32 characters long."
    )

# ---------------------------------------------------------
# 3. Initialise FastAPI
# ---------------------------------------------------------

inference_app = FastAPI(
    title="NaijaCyber N-ATLaS Inference",
    version="0.1.0"
)

generation_lock = threading.Lock()


class ChatMessage(BaseModel):
    role: str
    content: str = Field(min_length=1, max_length=4000)


class ChatRequest(BaseModel):
    model: str
    messages: list[ChatMessage] = Field(
        min_length=1,
        max_length=8
    )
    max_tokens: int = Field(default=320, ge=1, le=400)
    temperature: float = Field(default=0.1, ge=0, le=1)


def require_auth(authorization: str | None):
    expected = f"Bearer {API_KEY}"

    if not authorization or not hmac.compare_digest(
        authorization, expected
    ):
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )


@inference_app.get("/health")
def health(authorization: str | None = Header(None)):
    require_auth(authorization)

    return {
        "status": "ready",
        "model": MODEL_ID,
        "gpu": torch.cuda.get_device_name(0)
    }


@inference_app.post("/v1/chat/completions")
def chat_completion(
    request: ChatRequest,
    authorization: str | None = Header(None)
):
    require_auth(authorization)

    if request.model != MODEL_ID:
        raise HTTPException(
            status_code=400,
            detail="Unsupported model identifier"
        )

    if not request.messages:
        raise HTTPException(
            status_code=422,
            detail="Messages required"
        )

    allowed_roles = {"system", "user", "assistant"}

    if any(
        msg.role not in allowed_roles
        for msg in request.messages
    ):
        raise HTTPException(
            status_code=422,
            detail="Invalid message role"
        )

    if request.messages[-1].role != "user":
        raise HTTPException(
            status_code=422,
            detail="Last message must be from the user"
        )

    # Only one inference request at a time to reduce
    # GPU memory pressure on Colab's T4.
    if not generation_lock.acquire(blocking=False):
        raise HTTPException(
            status_code=429,
            detail="Model busy. Try again shortly."
        )

    try:
        messages = [
            {
                "role": message.role,
                "content": message.content
            }
            for message in request.messages
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            date_string=datetime.now().strftime(
                "%d %b %Y"
            )
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            add_special_tokens=False
        ).to(model.device)

        # Respect the model's practical context limit.
        if inputs["input_ids"].shape[-1] + request.max_tokens > 8092:
            raise HTTPException(
                status_code=422,
                detail="Prompt exceeds context budget"
            )

        torch.cuda.synchronize()
        started = time.perf_counter()

        generation_kwargs = {
            "max_new_tokens": request.max_tokens,
            "repetition_penalty": 1.12,
            "pad_token_id": tokenizer.eos_token_id,
            "use_cache": True
        }

        if request.temperature == 0:
            generation_kwargs["do_sample"] = False
        else:
            generation_kwargs["do_sample"] = True
            generation_kwargs["temperature"] = (
                request.temperature
            )

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                **generation_kwargs
            )

        torch.cuda.synchronize()

        elapsed = time.perf_counter() - started

        generated_tokens = outputs[0][
            inputs["input_ids"].shape[-1]:
        ]

        answer = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        ).strip()

        return {
            "id": "naijacyber-colab",
            "object": "chat.completion",
            "model": MODEL_ID,
            "choices": [
                {
                    "index": 0,
                    "message": {
                        "role": "assistant",
                        "content": answer
                    },
                    "finish_reason": "stop"
                }
            ],
            "usage": {
                "prompt_tokens": int(
                    inputs["input_ids"].shape[-1]
                ),
                "completion_tokens": int(
                    generated_tokens.shape[-1]
                )
            },
            "inference_seconds": round(elapsed, 2)
        }

    finally:
        generation_lock.release()


print("N-ATLaS inference API configured.")
print("Model:", MODEL_ID)
print("Authentication: Bearer token required")


N-ATLaS inference API configured.
Model: NCAIR1/N-ATLaS
Authentication: Bearer token required


In [15]:

# =========================================================
# NAIJACYBER AI - START INFERENCE SERVER + HTTPS TUNNEL
# =========================================================

import time
import threading
import httpx

from google.colab import userdata
from pyngrok import ngrok

# 1. Start authenticated local FastAPI server
def run_inference_server():
    uvicorn.run(
        inference_app,
        host="127.0.0.1",
        port=8001,
        log_level="warning",
        access_log=False
    )

server_thread = threading.Thread(
    target=run_inference_server,
    daemon=True
)

server_thread.start()

# 2. Wait until the server responds
for attempt in range(30):
    try:
        response = httpx.get(
            "http://127.0.0.1:8001/health",
            timeout=2
        )

        # 401 confirms the server is running and
        # authentication is enforced.
        if response.status_code == 401:
            break

    except httpx.RequestError:
        pass

    time.sleep(1)
else:
    raise RuntimeError(
        "Inference server did not start."
    )

# 3. Configure ngrok
NGROK_AUTHTOKEN = userdata.get(
    "NGROK_AUTHTOKEN"
)

if not NGROK_AUTHTOKEN:
    raise RuntimeError(
        "NGROK_AUTHTOKEN is missing from Colab Secrets."
    )

ngrok.set_auth_token(NGROK_AUTHTOKEN)

# 4. Create temporary HTTPS tunnel
tunnel = ngrok.connect(
    addr=8001,
    proto="http"
)

public_url = tunnel.public_url

if not public_url.startswith("https://"):
    ngrok.disconnect(public_url)
    raise RuntimeError(
        "Expected an HTTPS ngrok tunnel."
    )

# 5. Display the endpoint
print("=" * 55)
print("NAIJACYBER AI - N-ATLaS INFERENCE READY")
print("=" * 55)
print("Public tunnel:", public_url)
print(
    "Chat endpoint:",
    f"{public_url}/v1/chat/completions"
)
print("Authentication: Required")
print("Model:", MODEL_ID)


NAIJACYBER AI - N-ATLaS INFERENCE READY
Public tunnel: [temporary tunnel URL redacted]
Chat endpoint: [temporary tunnel URL redacted]
Authentication: Required
Model: NCAIR1/N-ATLaS


In [16]:
import httpx

from google.colab import userdata

api_key = userdata.get("NATLAS_API_KEY")

# Unauthenticated request should be rejected
unauthorized = httpx.get(
    f"{public_url}/health",
    timeout=15
)

print(
    "Without authentication:",
    unauthorized.status_code
)

# Authenticated request should succeed
authorized = httpx.get(
    f"{public_url}/health",
    headers={
        "Authorization": f"Bearer {api_key}"
    },
    timeout=15
)

print(
    "With authentication:",
    authorized.status_code
)

print(authorized.json())


Without authentication: 401
With authentication: 200
{'status': 'ready', 'model': 'NCAIR1/N-ATLaS', 'gpu': 'Tesla T4'}


In [17]:
import httpx
from google.colab import userdata

secret = userdata.get("NATLAS_API_KEY")

response = httpx.get(
    "http://127.0.0.1:8001/health",
    headers={
        "Authorization": f"Bearer {secret}"
    },
    timeout=10
)

print("Local authentication status:", response.status_code)
print("Response:", response.json())

Local authentication status: 200
Response: {'status': 'ready', 'model': 'NCAIR1/N-ATLaS', 'gpu': 'Tesla T4'}


In [18]:
import httpx
from google.colab import userdata

secret = userdata.get("NATLAS_API_KEY")

response = httpx.get(
    f"{public_url}/health",
    headers={
        "Authorization": f"Bearer {secret}"
    },
    timeout=20,
    follow_redirects=False
)

print("Public authentication status:", response.status_code)

if response.status_code == 200:
    print("Authenticated connection successful.")
    print(response.json())
else:
    print("Authentication or tunnel issue.")
    print("Content type:", response.headers.get("content-type"))

Public authentication status: 200
Authenticated connection successful.
{'status': 'ready', 'model': 'NCAIR1/N-ATLaS', 'gpu': 'Tesla T4'}


In [19]:
import hashlib
from google.colab import userdata

secret = userdata.get("NATLAS_API_KEY")

fingerprint = hashlib.sha256(
    secret.encode("utf-8")
).hexdigest()[:12]

print("Colab API key fingerprint:", fingerprint)

Colab API key fingerprint: [redacted]


In [20]:
import hashlib
import hmac
from google.colab import userdata

current_secret = userdata.get("NATLAS_API_KEY")

print(
    "Running server key matches Colab Secret:",
    hmac.compare_digest(API_KEY, current_secret)
)

Running server key matches Colab Secret: True


In [21]:
import httpx
from google.colab import userdata

secret = userdata.get("NATLAS_API_KEY")

headers = {
    "Authorization": f"Bearer {secret}"
}

# Test the local Colab server
local_response = httpx.get(
    "http://127.0.0.1:8001/health",
    headers=headers,
    timeout=15
)

print("LOCAL COLAB:", local_response.status_code)

# Test through the public ngrok tunnel
public_response = httpx.get(
    f"{public_url}/health",
    headers=headers,
    timeout=20,
    follow_redirects=False
)

print("PUBLIC NGROK:", public_response.status_code)

if public_response.status_code != 200:
    print(
        "Public response type:",
        public_response.headers.get("content-type")
    )

LOCAL COLAB: 200
PUBLIC NGROK: 200
